# Experiment 3: ACS 2023 target and ACS 2014 auxiliary shadows
Train the target on 100,000 California 2023 records with expanded predictors.
Compare 16 California 2014 shadows with 16 UCI Adult shadows in separate cells.
Both pools cycle random forest (6), gradient boosting (5), and XGBoost (5).
Each model fits its own preprocessing on its training partition only.
The label is nominal personal income (PINCP) >$50,000 in both years, without
ADJINC or CPI inflation adjustment. Original eligibility filters are retained. Occupation and other category
codes remain native to each year; no cross-year code equivalence is assumed.
Income-derived predictors, identifiers, survey weights and allocation flags are
excluded. MIA uses logit, blind walk, entropy and cross-entropy.
Perturbation calls retain the original protocol: categorical commuting changes
for shadows and numeric commuting changes for the target. No protocol correction
was applied in the 100,000-record run.
Prepared for review; execution requires user approval.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.data import load_acs_income, split_acs_data, subset_target_training
from src.expanded_data import (
    load_expanded_acs, preprocess_expanded_partitions, expanded_feature_inventory,
    reference_split_config, EXPANDED_FEATURES, EXCLUDED, load_expanded_year,
)
from src.training import train_target_model, evaluate_model
from src.exp3_checkpoints import cached_target_train, cached_expanded_shadows
from src.shadow_models import (load_adult_data, train_shadow_models,
                               summarize_shadow_models, PAPER_SHADOW_ALGORITHMS)
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import (collect_shadow_paper_features, collect_target_paper_features,
                                       make_probability_oracle)

torch.set_num_threads(4)
SEED = 42
TRAIN_SIZE = 100_000
INCOME_BASE_YEAR = None
NOMINAL_INCOME_METADATA = dict(enabled=False, label='PINCP > 50000', adjustment='None')
RUN_DIR = Path('artifacts/exp_3_2023_2014_100000_nominal_paper_algorithms')
ADULT_RUN_DIR = RUN_DIR / 'adult_auxiliary'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED, 'Use the reference seed consistently.'
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0, seed=SEED,
    device='cpu', early_stopping=False, restore_best=False, monitor_train=True)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=SEED)
SHADOW_RECORDS_PER_CLASS = 2000
N_SHADOW_MODELS = 16
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS
# Native commuting units; all other added codes are categorical in the walk.
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
print('Training size:', TRAIN_SIZE, '| Input fields:', len(EXPANDED_FEATURES))
print('Reference split configuration:', SPLIT_CONFIG)


Training size: 100000 | Input fields: 48
Reference split configuration: {'validation_size': 10000, 'test_size': 10000, 'mia_nonmember_size': 10000, 'mia_member_size': 10000, 'seed': 42}


## Load 2023 target records and make stratified partitions
Split sizes and seed follow Experiment 2, applied to the new 2023 records.


In [2]:
expanded_source = load_expanded_year(2023, income_base_year=INCOME_BASE_YEAR)
reference_partitions = split_acs_data(expanded_source, **SPLIT_CONFIG)
expanded_base_partitions = reference_partitions
expanded_partitions = subset_target_training(
    expanded_base_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(expanded_partitions[name].index, reference_partitions[name].index)
assert set(expanded_partitions['mia_members'].index).issubset(expanded_partitions['train'].index)
for name in ['validation', 'test', 'mia_nonmembers']:
    assert set(expanded_partitions[name].index).isdisjoint(expanded_partitions['train'].index)
print('Target income adjustment:', expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
print('Eligible records:', len(expanded_source))
print('Unused records:', len(expanded_source) - TRAIN_SIZE
      - sum(len(expanded_partitions[name]) for name in ['validation', 'test', 'mia_nonmembers']))
display(pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
    for name, part in expanded_partitions.items()}).T)
feature_inventory = expanded_feature_inventory(expanded_partitions['train'])
display(feature_inventory)


Target income adjustment: {'enabled': False, 'label': 'PINCP > 50000', 'adjustment': 'None'}
Eligible records: 203278
Unused records: 73278


,records,positive_rate
validation,10000.0,0.49840
test,10000.0,0.49840
mia_nonmembers,10000.0,0.49840
train,100000.0,0.49845
mia_members,10000.0,0.49840


,group,type,unique_nonmissing,missing_fraction
feature,,,,
AGEP,original_ACSIncome,numeric,75,0.00000
COW,original_ACSIncome,categorical,8,0.00000
SCHL,original_ACSIncome,categorical,24,0.00000
MAR,original_ACSIncome,categorical,5,0.00000
OCCP,original_ACSIncome,categorical,527,0.00000
POBP,original_ACSIncome,categorical,220,0.00000
WKHP,original_ACSIncome,numeric,97,0.00000
SEX,original_ACSIncome,categorical,2,0.00000
RAC1P,original_ACSIncome,categorical,9,0.00000


## Preprocess using training records only
The output is sparse; the MLP densifies only the current batch. Cardinality
counts above refer to the original fields, not one-hot dimensions.


In [3]:
expanded_preprocessor, expanded_arrays = preprocess_expanded_partitions(expanded_partitions)
print('Encoded input dimensions:', expanded_arrays['train']['X'].shape[1])
print('Training CSR storage (MiB):', round(sum(
    value.nbytes for value in [expanded_arrays['train']['X'].data,
    expanded_arrays['train']['X'].indices, expanded_arrays['train']['X'].indptr]) / 2**20, 2))
pd.DataFrame({name: {'rows': values['X'].shape[0], 'encoded_features': values['X'].shape[1]}
    for name, values in expanded_arrays.items()}).T


Encoded input dimensions: 2689
Training CSR storage (MiB): 38.53


,rows,encoded_features
validation,10000,2689
test,10000,2689
mia_nonmembers,10000,2689
train,100000,2689
mia_members,10000,2689


## Train the target
Fixed 100 epochs, final weights retained. Validation is monitored but does not
select an epoch in this deliberately overfit experimental condition.


In [4]:
target_started = time.monotonic()
expanded_model, expanded_history, expanded_metadata = cached_target_train(
    expanded_arrays, TRAIN_CONFIG, RUN_DIR)
target_training_seconds = time.monotonic() - target_started
print('Target training seconds:', round(target_training_seconds, 1))
display(pd.DataFrame(expanded_history).iloc[::10])


Epoch 001: train loss=0.4165, validation loss=0.4039, validation accuracy=0.8134


Epoch 002: train loss=0.3795, validation loss=0.3929, validation accuracy=0.8213


Epoch 003: train loss=0.3570, validation loss=0.4011, validation accuracy=0.8173


Epoch 004: train loss=0.3249, validation loss=0.4136, validation accuracy=0.8132


Epoch 005: train loss=0.2825, validation loss=0.4479, validation accuracy=0.8089


Epoch 006: train loss=0.2292, validation loss=0.5128, validation accuracy=0.8071


... (92 outputs left out)

Epoch 099: train loss=0.0028, validation loss=1.8994, validation accuracy=0.7876


Epoch 100: train loss=0.0016, validation loss=1.9594, validation accuracy=0.7865
Completed target checkpoint saved.


Target training seconds: 885.6


,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss,train_eval_accuracy,train_eval_roc_auc,train_eval_log_loss
0,1,0.416451,0.8134,0.900563,0.403865,0.82923,0.913752,0.378947
10,11,0.060908,0.7923,0.867702,1.158196,0.98532,0.999131,0.039848
20,21,0.016433,0.7787,0.858233,1.512833,0.99233,0.999692,0.021763
30,31,0.011112,0.7940,0.868656,1.540580,0.99794,0.999976,0.006108
40,41,0.007754,0.7937,0.868837,1.735596,0.99766,0.999973,0.006455
50,51,0.007198,0.7898,0.869194,1.748980,0.99778,0.999985,0.005592
60,61,0.004103,0.7855,0.862731,1.791714,0.99801,0.999974,0.006090
70,71,0.005068,0.7877,0.863999,1.846895,0.99766,0.999985,0.006761
80,81,0.003399,0.7911,0.867223,1.839046,0.99867,0.999990,0.003930
90,91,0.003514,0.7906,0.865856,1.933827,0.99936,0.999998,0.001810


## Income-prediction evaluation
These are target income metrics, not membership metrics. The test set was not
used for model selection. Do not choose target settings using these results.


In [5]:
expanded_metrics = {name: evaluate_model(expanded_model, expanded_arrays[name]['X'],
    expanded_arrays[name]['y']) for name in ['train', 'validation', 'test']}
display(pd.DataFrame(expanded_metrics).T)
print('Train-test accuracy gap (percentage points):',
      100 * (expanded_metrics['train']['accuracy'] - expanded_metrics['test']['accuracy']))


,accuracy,roc_auc,log_loss
train,0.99849,0.999985,0.004646
validation,0.78650,0.861110,1.959403
test,0.79520,0.867629,1.864808


Train-test accuracy gap (percentage points): 20.328999999999997


## 16 expanded ACS 2014 shadows using the paper's three named algorithms
Keep existing RF/XGBoost settings and 40/20/40 splits.
Gradient boosting uses 300 trees, learning rate 0.05, maximum depth 3, seed per shadow.
These are project settings; the paper does not provide its exact configurations.
Each preprocessor fits its own training records.
These shadows do not assume knowledge of the target architecture.
Cycle the three algorithms using unique seeds and splits. Models may reuse auxiliary
records across shadows; grouped attack splits keep repeated records together.


In [6]:
acs_auxiliary = load_expanded_year(2014, income_base_year=INCOME_BASE_YEAR)
print('Auxiliary income adjustment:', acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
assert set(acs_auxiliary.index).isdisjoint(expanded_source.index)
shadow_results = cached_expanded_shadows(acs_auxiliary, seed=SEED, n_models=N_SHADOW_MODELS,
    run_dir=RUN_DIR, algorithms=SHADOW_ALGORITHMS)
# Verify native-input queries reproduce probabilities computed during training.
for name, result in shadow_results.items():
    audit_rows = result['attack_data'].iloc[:32]
    native = acs_auxiliary.loc[audit_rows['record_id'], EXPANDED_FEATURES]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema='acs')
    np.testing.assert_allclose(query(native),
        audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(),
        rtol=1e-5, atol=1e-6, err_msg=f'Query mismatch for {name}')
print('Probability oracle consistency verified for all 16 shadows.')
display(summarize_shadow_models(shadow_results))


Auxiliary income adjustment: {'enabled': False, 'label': 'PINCP > 50000', 'adjustment': 'None'}


Training shadow 1/16: random_forest_01


Completed shadow checkpoint saved: random_forest_01


Training shadow 2/16: gradient_boosting_02


Completed shadow checkpoint saved: gradient_boosting_02


Training shadow 3/16: xgboost_03


... (25 outputs left out)

Training shadow 16/16: random_forest_16


Completed shadow checkpoint saved: random_forest_16


Probability oracle consistency verified for all 16 shadows.


records  accuracy   roc_auc  log_loss
shadow_model         partition                                        
random_forest_01     train         73576  0.895034  0.963991  0.305143
                     validation    36789  0.822447  0.898100  0.396220
                     nonmembers    73576  0.823815  0.899010  0.395105
gradient_boosting_02 train         73576  0.828857  0.905512  0.374626
                     validation    36789  0.828046  0.902338  0.379613
                     nonmembers    73576  0.825092  0.902493  0.379359
xgboost_03           train         73576  0.841076  0.916833  0.352353
                     validation    36789  0.834489  0.911705  0.360892
                     nonmembers    73576  0.833669  0.909643  0.364265
random_forest_04     train         73576  0.895876  0.964020  0.306229
                     validation    36789  0.826307  0.900072  0.393914
                     nonmembers    73576  0.824019  0.898980  0.395291
gradient_boosting_05 train         73576  0.828776  0.906059  0.373634
                     validation    36789  0.826307  0.903606  0.377615
                     nonmembers    73576  0.826696  0.902084  0.379925
xgboost_06           train         73576  0.843645  0.918923  0.348288
                     validation    36789  0.833592  0.909114  0.364757
                     nonmembers    73576  0.832255  0.908488  0.365835
random_forest_07     train         73576  0.893851  0.963988  0.304787
                     validation    36789  0.824350  0.900185  0.392477
                     nonmembers    73576  0.821137  0.897831  0.396558
gradient_boosting_08 train         73576  0.830842  0.907635  0.370956
                     validation    36789  0.823942  0.901067  0.381202
                     nonmembers    73576  0.824603  0.901087  0.381115
xgboost_09           train         73576  0.842326  0.918027  0.349982
                     validation    36789  0.833374  0.907983  0.366911
                     nonmembers    73576  0.834267  0.910003  0.363418
random_forest_10     train         73576  0.895496  0.963504  0.305802
                     validation    36789  0.824621  0.896990  0.397782
                     nonmembers    73576  0.826248  0.901203  0.392587
gradient_boosting_11 train         73576  0.828205  0.905770  0.374374
                     validation    36789  0.824051  0.901651  0.380322
                     nonmembers    73576  0.826248  0.902440  0.379078
xgboost_12           train         73576  0.841538  0.918526  0.349140
                     validation    36789  0.834869  0.909549  0.364243
                     nonmembers    73576  0.832432  0.908971  0.364974
random_forest_13     train         73576  0.893077  0.963590  0.305655
                     validation    36789  0.823453  0.900789  0.392610
                     nonmembers    73576  0.823665  0.898826  0.394883
gradient_boosting_14 train         73576  0.828545  0.905773  0.374242
                     validation    36789  0.825437  0.901572  0.380655
                     nonmembers    73576  0.825894  0.902501  0.379467
xgboost_15           train         73576  0.842394  0.918390  0.349351
                     validation    36789  0.833836  0.908037  0.366705
                     nonmembers    73576  0.832432  0.908918  0.364909
random_forest_16     train         73576  0.894626  0.964139  0.305157
                     validation    36789  0.825682  0.899839  0.394492
                     nonmembers    73576  0.824046  0.899143  0.394792

## Extract shadow features and fit the random forest attack
Features: class-scaled logit, blind walk, entropy and cross-entropy. Use 2,000
members and 2,000 non-members per shadow. Attack record-ID grouping prevents
repeated 2014 records from crossing attack partitions. Select both thresholds
on auxiliary validation only; no target membership labels train this classifier.


In [7]:
shadow_attack_data, shadow_query_diagnostics = collect_shadow_paper_features(
    shadow_results, acs_auxiliary, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=SEED)
display(pd.DataFrame(shadow_query_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper')
display(pd.DataFrame(attack_result['metrics']).T)


Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (120 outputs left out)

Attack feature extraction: 3,584/4,000


Attack feature extraction: 4,000/4,000


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
random_forest_01,126429,164000,0.643375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_02,124549,164000,0.613375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_03,124439,164000,0.616375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_04,126876,164000,0.647,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_05,123461,164000,0.60925,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_06,124575,164000,0.609,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_07,127145,164000,0.645875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_08,123744,164000,0.609125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_09,124469,164000,0.615125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_10,127005,164000,0.647125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,38419.0,19317.0,0.601942,0.601382,0.646115,0.032769,0.507089,9577.0,9525.0,5768.0,13549.0,0.596605,0.015945,0.004031
validation,12849.0,6376.0,0.519496,0.520273,0.521083,0.012704,0.507089,2701.0,3772.0,2402.0,3974.0,0.596605,0.012704,0.009887
test,12732.0,6307.0,0.511074,0.512006,0.520823,0.012526,0.507089,2644.0,3781.0,2444.0,3863.0,0.596605,0.011099,0.009183


## Final MIA evaluation on the expanded target
Query the expanded native input schema, preserving categorical codes and
handling numeric commute fields in native units. The walk still generates
field-valid but potentially implausible combinations, and its dimensionality
matches the expanded auxiliary schema. Thresholds are frozen from auxiliary validation.


In [8]:
target_attack_data, target_query_diagnostics = collect_target_paper_features(
    expanded_model, expanded_preprocessor, expanded_partitions,
    config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
display(pd.Series(target_query_diagnostics, name='expanded target query diagnostics'))
target_mia_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
target_mia_metrics['tpr_at_accuracy_threshold'] = target_mia_metrics['tp'] / target_mia_metrics['members']
target_mia_metrics['fpr_at_accuracy_threshold'] = target_mia_metrics['fp'] / (
    target_mia_metrics['records'] - target_mia_metrics['members'])
display(pd.DataFrame({'2014 auxiliary attack test': attack_result['metrics']['test'],
                      'Expanded ACS target MIA': target_mia_metrics}).T)


Attack feature extraction: 512/20,000


Attack feature extraction: 1,024/20,000


Attack feature extraction: 1,536/20,000


Attack feature extraction: 2,048/20,000


Attack feature extraction: 2,560/20,000


Attack feature extraction: 3,072/20,000


... (32 outputs left out)

Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


record_queries                                                       526991
max_record_queries                                                   820000
walk_censored_fraction                                             0.438975
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: expanded target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold
2014 auxiliary attack test,12732.0,6307.0,0.511074,0.512006,0.520823,0.012526,0.507089,2644.0,3781.0,2444.0,3863.0,0.596605,0.011099,0.009183,NaN,NaN
Expanded ACS target MIA,20000.0,10000.0,0.626250,0.626250,0.608622,0.001700,0.507089,2724.0,7276.0,199.0,9801.0,0.596605,0.001400,0.004500,0.9801,0.7276


## Small reproducibility artifacts
Save summaries, feature inventory and row assignments. Completed target and
shadow checkpoints are saved during training to support recovery after interruption.
Source CSVs and the other experiment notebooks remain unchanged.


In [9]:
result_dir = RUN_DIR
result_dir.mkdir(parents=True, exist_ok=True)
feature_inventory.to_csv(result_dir / 'feature_inventory.csv')
for name, part in expanded_partitions.items():
    pd.Series(part.index, name='record_id').to_csv(result_dir / f'{name}_ids.csv', index=False)
summarize_shadow_models(shadow_results).to_csv(result_dir / 'shadow_metrics.csv')
summary = dict(income_base_year=INCOME_BASE_YEAR,
    target_income_adjustment=expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    auxiliary_income_adjustment=acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    perturbation_protocol_audit={'shadow_numeric_fields': {'AGEP': [1.0, 17, 99], 'WKHP': [2.0, 1, 99]}, 'target_extra_numeric_fields': {'JWMNP': [2.0, 0, 200], 'JWRIP': [1.0, 1, 10]}, 'shadow_commuting_fields': 'Categorical replacements, unchanged from original run', 'target_commuting_fields': 'Numeric perturbations, unchanged from original run', 'correction': 'Prior assistant statements that the shadow commuting perturbation protocol changed were incorrect. Original and current perturbation call sources are identical.'},
    walk_numeric_protocol_scope='Extra target numeric fields only',
    walk_numeric_protocol=EXPANDED_NUMERIC_WALK, execution_status='all_cells_completed',
    attack_metrics_status='oracle_consistency_verified', target_year=2023, auxiliary_year=2014, n_shadow_models=N_SHADOW_MODELS,
    shadow_algorithms=list(SHADOW_ALGORITHMS),
    shadow_seeds={name: result['seed'] for name, result in shadow_results.items()},
    original_feature_count=len(EXPANDED_FEATURES),
    encoded_feature_count=expanded_arrays['train']['X'].shape[1],
    train_size=TRAIN_SIZE, split_config=SPLIT_CONFIG, training=expanded_metadata,
    target_training_seconds=target_training_seconds, target_metrics=expanded_metrics,
    shadow_attack_metrics=attack_result['metrics'], target_mia_metrics=target_mia_metrics,
    target_query_diagnostics=target_query_diagnostics)
(result_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Summary and exact row IDs written to', result_dir)


Summary and exact row IDs written to artifacts\exp_3_2023_2014_100000_nominal_paper_algorithms


## UCI Adult auxiliary comparison: load every record and train 16 shadows
Pool the original Adult training and test files as auxiliary information only.
Use all 48,842 rows: per shadow, 40% training members, 20% validation and 40%
held-out non-members. Partitions are disjoint within each shadow. Across shadows,
records can be reused; grouped attack splits prevent record leakage.
Adult uses its native 12 predictors and nominal income label, not a fabricated
mapping to the 48 ACS predictors. The same four output/query features transfer
to the same ACS target. Differences in input geometry and population limit comparison.
No Adult feature-extraction subsampling is used. This increases query cost.


In [10]:
adult_auxiliary = load_adult_data()
assert len(adult_auxiliary) == 48_842, 'Expected complete UCI Adult dataset.'
assert adult_auxiliary.index.is_unique
assert set(adult_auxiliary.index).isdisjoint(expanded_source.index)
print('Adult auxiliary records:', len(adult_auxiliary))
adult_shadow_results = cached_expanded_shadows(
    adult_auxiliary, seed=SEED, n_models=N_SHADOW_MODELS,
    run_dir=ADULT_RUN_DIR, algorithms=SHADOW_ALGORITHMS, schema='adult')
for name, result in adult_shadow_results.items():
    assigned = [set(ids) for ids in result['partition_ids'].values()]
    assert sum(map(len, assigned)) == len(adult_auxiliary)
    assert set.union(*assigned) == set(adult_auxiliary.index)
    assert all(a.isdisjoint(b) for i, a in enumerate(assigned) for b in assigned[i+1:])
    audit_rows = result['attack_data'].iloc[:32]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema='adult')
    np.testing.assert_allclose(query(adult_auxiliary.loc[audit_rows['record_id']]),
        audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(), rtol=1e-5, atol=1e-6)
display(summarize_shadow_models(adult_shadow_results))


Adult auxiliary records: 48842
Training shadow 1/16: random_forest_01


Completed shadow checkpoint saved: random_forest_01


Training shadow 2/16: gradient_boosting_02


Completed shadow checkpoint saved: gradient_boosting_02


Training shadow 3/16: xgboost_03


Completed shadow checkpoint saved: xgboost_03


... (23 outputs left out)

Completed shadow checkpoint saved: xgboost_15


Training shadow 16/16: random_forest_16


Completed shadow checkpoint saved: random_forest_16


records  accuracy   roc_auc  log_loss
shadow_model         partition                                        
random_forest_01     train         19536  0.895219  0.959761  0.244717
                     validation     9769  0.868769  0.917441  0.302059
                     nonmembers    19537  0.861084  0.915341  0.308461
gradient_boosting_02 train         19536  0.871929  0.928143  0.279730
                     validation     9769  0.868359  0.926754  0.282966
                     nonmembers    19537  0.867738  0.921267  0.291666
xgboost_03           train         19536  0.883446  0.941305  0.255014
                     validation     9769  0.871839  0.926513  0.281488
                     nonmembers    19537  0.872089  0.927904  0.278240
random_forest_04     train         19536  0.893376  0.958961  0.245891
                     validation     9769  0.862934  0.918304  0.302124
                     nonmembers    19537  0.862824  0.917640  0.302903
gradient_boosting_05 train         19536  0.876280  0.932757  0.272778
                     validation     9769  0.867335  0.923023  0.288946
                     nonmembers    19537  0.867841  0.921961  0.289852
xgboost_06           train         19536  0.884726  0.941953  0.254491
                     validation     9769  0.873989  0.929190  0.275766
                     nonmembers    19537  0.870297  0.924533  0.283342
random_forest_07     train         19536  0.895321  0.961814  0.240577
                     validation     9769  0.865493  0.917681  0.304088
                     nonmembers    19537  0.859702  0.913600  0.308120
gradient_boosting_08 train         19536  0.873311  0.930350  0.277673
                     validation     9769  0.864162  0.920147  0.292709
                     nonmembers    19537  0.869683  0.924202  0.286750
xgboost_09           train         19536  0.885596  0.942453  0.253310
                     validation     9769  0.871839  0.925299  0.282861
                     nonmembers    19537  0.871628  0.927458  0.279251
random_forest_10     train         19536  0.897727  0.960232  0.244341
                     validation     9769  0.864367  0.918858  0.300944
                     nonmembers    19537  0.862927  0.916029  0.304557
gradient_boosting_11 train         19536  0.876894  0.931503  0.274500
                     validation     9769  0.867028  0.919172  0.294309
                     nonmembers    19537  0.866663  0.922411  0.289391
xgboost_12           train         19536  0.886927  0.943644  0.250436
                     validation     9769  0.864674  0.922922  0.286783
                     nonmembers    19537  0.871321  0.925134  0.282485
random_forest_13     train         19536  0.893837  0.959454  0.244999
                     validation     9769  0.864572  0.917861  0.301353
                     nonmembers    19537  0.864616  0.918486  0.302652
gradient_boosting_14 train         19536  0.874027  0.931907  0.274190
                     validation     9769  0.866619  0.922466  0.289202
                     nonmembers    19537  0.868250  0.921208  0.291364
xgboost_15           train         19536  0.884521  0.942658  0.252899
                     validation     9769  0.871123  0.925261  0.283502
                     nonmembers    19537  0.871884  0.926159  0.280792
random_forest_16     train         19536  0.896499  0.960432  0.242556
                     validation     9769  0.861194  0.913233  0.308263
                     nonmembers    19537  0.862057  0.914470  0.306895

## Adult attack features: all available members and non-members
Validation rows retain their existing model-validation role. Every training member
and held-out non-member contributes attack features (per_class=None).
Use the existing native Adult perturbation protocol and unchanged RF attack.
Thresholds are selected on auxiliary attack validation only.


In [11]:
adult_attack_data, adult_query_diagnostics = collect_shadow_paper_features(
    adult_shadow_results, adult_auxiliary, per_class=None, config=WALK_CONFIG, seed=SEED)
assert len(adult_attack_data) == sum(
    len(result['attack_data']) for result in adult_shadow_results.values())
display(adult_attack_data.groupby(['shadow_model', 'membership']).size().unstack())
adult_attack_result = train_membership_classifier(
    adult_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper')
display(pd.DataFrame(adult_attack_result['metrics']).T)


Attack feature extraction: 512/39,073


Attack feature extraction: 1,024/39,073


Attack feature extraction: 1,536/39,073


Attack feature extraction: 2,048/39,073


Attack feature extraction: 2,560/39,073


Attack feature extraction: 3,072/39,073


... (1,224 outputs left out)

Attack feature extraction: 38,912/39,073


Attack feature extraction: 39,073/39,073


membership,0,1
shadow_model,,
gradient_boosting_02,19537,19536
gradient_boosting_05,19537,19536
gradient_boosting_08,19537,19536
gradient_boosting_11,19537,19536
gradient_boosting_14,19537,19536
random_forest_01,19537,19536
random_forest_04,19537,19536
random_forest_07,19537,19536
random_forest_10,19537,19536


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,375206.0,187275.0,0.537454,0.537902,0.570357,0.021898,0.496537,52949.0,134982.0,38568.0,148707.0,0.559061,0.015176,0.006119
validation,125200.0,62895.0,0.515599,0.514394,0.519558,0.010653,0.496537,16121.0,46184.0,14463.0,48432.0,0.559061,0.010653,0.009999
test,124762.0,62406.0,0.514179,0.514078,0.519221,0.011682,0.496537,16405.0,45951.0,14661.0,47745.0,0.559061,0.012050,0.010232


## Compare both auxiliary sources against the same saved target queries
Reuse target_attack_data from above: no retraining or additional target queries.
Both attacks use thresholds frozen on their own auxiliary validation data.


In [12]:
adult_target_mia_metrics, adult_target_scores = evaluate_attack(
    adult_attack_result['classifier'], target_attack_data,
    threshold=adult_attack_result['accuracy_threshold'],
    low_fpr_threshold=adult_attack_result['low_fpr_threshold'])
comparison = pd.DataFrame({
    'ACS 2014 auxiliary': target_mia_metrics,
    'UCI Adult auxiliary (all records)': adult_target_mia_metrics}).T
display(comparison)
ADULT_RUN_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RUN_DIR / 'auxiliary_comparison.csv')
summarize_shadow_models(adult_shadow_results).to_csv(ADULT_RUN_DIR / 'shadow_metrics.csv')
adult_summary = dict(execution_status='all_adult_cells_completed', auxiliary='UCI Adult',
    auxiliary_records=len(adult_auxiliary), shadow_schema='adult',
    n_shadow_models=N_SHADOW_MODELS, algorithms=list(SHADOW_ALGORITHMS),
    split_fractions=dict(train=0.4, validation=0.2, nonmembers=0.4),
    feature_sampling_per_membership_class=None,
    shadow_seeds={name: result['seed'] for name, result in adult_shadow_results.items()},
    shadow_attack_metrics=adult_attack_result['metrics'],
    target_mia_metrics=adult_target_mia_metrics, query_diagnostics=adult_query_diagnostics,
    target_summary='../summary.json', target_queries_reused=True)
(ADULT_RUN_DIR / 'summary.json').write_text(json.dumps(adult_summary, indent=2), encoding='utf-8')
print('Adult comparison saved to', ADULT_RUN_DIR)


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold
ACS 2014 auxiliary,20000.0,10000.0,0.62625,0.62625,0.608622,0.0017,0.507089,2724.0,7276.0,199.0,9801.0,0.596605,0.0014,0.0045,0.9801,0.7276
UCI Adult auxiliary (all records),20000.0,10000.0,0.55060,0.55060,0.567290,0.0033,0.496537,4331.0,5669.0,3319.0,6681.0,0.559061,0.0008,0.0036,NaN,NaN


Adult comparison saved to artifacts\exp_3_2023_2014_100000_nominal_paper_algorithms\adult_auxiliary
